El proyecto llamado Ghost Scout es un sistema que utiliza algoritmos KNN (vecinos mas cercanos) para identificar perfiles similares
de futbolistas basados en su rendimiento estadistico p90.

En este notebook se encuentra la primera fase del proyecto donde aprendo como manejar los datos de futbol, convertirlos a p90 y todas
las dificultades que conllevan el tratamientode los eventos masivos.

En siguientes notebooks se planea aplicar mejoras como:
    -Agregar valores que puedan representar mas fielmente el desarrollo del juego (varaibles por tercio del campo, xG, xT, etc).
    -Utilizar valores actuales a traves de web scrapping y poder hacer comparativa cada fin de semana.
    -Integrar una LLM para hacer informes de aquellos jugadores que se parezcan mas a las estrellas del futbol mundial.

In [1]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
from statsbombpy import sb
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json

In [2]:
import os

# Verificar si el archivo ya existe para evitar descargarlo nuevamente
if os.path.exists('data/raw/eventos_laliga_2020_2021.parquet'):
    df_events = pd.read_parquet('data/raw/eventos_laliga_2020_2021.parquet')
    print('Descargado desde el archivo local')

# Si el archivo no existe, descargarlo desde StatsBomb y guardarlo localmente
# se va metiendo cada partido en un dataframe y luego se concatena todos los partidos en un solo dataframe
else:
    partidos = sb.matches(competition_id=11, season_id=90)
    match_ids = partidos['match_id'].tolist()

    eventos = []
    for match_id in match_ids:
        df_match = sb.events(match_id=match_id,flatten_attrs=True)
        eventos.append(df_match)

    df_events = pd.concat(eventos, ignore_index=True)
    df_events.to_parquet('data/raw/eventos_laliga_2020_2021.parquet', index=False)
    print('Descargado desde StatsBomb y guardado localmente')


print(f"Tamaño del DataFrame: {df_events.shape}")

Descargado desde el archivo local
Tamaño del DataFrame: (139030, 111)


In [3]:
# Mostrar las primeras filas del DataFrame
df_events.head()

,50_50,bad_behaviour_card,ball_receipt_outcome,ball_recovery_offensive,ball_recovery_recovery_failure,carry_end_location,clearance_aerial_won,clearance_body_part,clearance_head,clearance_left_foot,...,foul_won_penalty,goalkeeper_punched_out,goalkeeper_shot_saved_off_target,goalkeeper_shot_saved_to_post,shot_saved_off_target,shot_saved_to_post,block_save_block,dribble_no_touch,shot_redirect,shot_follows_dribble
0,None,NaN,NaN,None,None,None,None,NaN,None,None,...,None,None,None,None,None,None,None,None,None,None
1,None,NaN,NaN,None,None,None,None,NaN,None,None,...,None,None,None,None,None,None,None,None,None,None
2,None,NaN,NaN,None,None,None,None,NaN,None,None,...,None,None,None,None,None,None,None,None,None,None
3,None,NaN,NaN,None,None,None,None,NaN,None,None,...,None,None,None,None,None,None,None,None,None,None
4,None,NaN,NaN,None,None,None,None,NaN,None,None,...,None,None,None,None,None,None,None,None,None,None


hay muchas columnas y muchos datos para ver informacion solo con las primeras filas

In [4]:
# Mostrar información general del DataFrame
df_events.describe()

,duration,index,match_id,minute,pass_angle,pass_length,pass_recipient_id,period,player_id,possession,possession_team_id,second,shot_statsbomb_xg,substitution_outcome_id,substitution_replacement_id,team_id
count,99833.000000,139030.000000,1.390300e+05,139030.000000,40337.000000,40337.000000,39210.000000,139030.000000,138548.000000,139030.000000,139030.000000,139030.000000,839.000000,300.000000,300.000000,139030.000000
mean,1.243923,1994.824347,3.773010e+06,44.716025,0.011773,19.008731,12587.533563,1.494282,12990.401969,84.916903,272.851809,29.247249,0.128030,102.940000,17170.006667,278.560073
std,1.198001,1160.903654,2.113479e+03,27.028860,1.613208,12.670579,11184.041337,0.499969,11358.324489,51.232888,188.171547,17.350284,0.166607,0.237884,14638.071250,196.319253
min,0.000000,1.000000,3.764440e+06,0.000000,-3.136465,0.000000,2948.000000,1.000000,2948.000000,1.000000,206.000000,0.000000,0.003424,102.000000,3163.000000,206.000000
25%,0.489306,994.000000,3.773415e+06,21.000000,-1.306833,10.610373,5477.000000,1.000000,5487.000000,42.000000,217.000000,14.000000,0.037838,103.000000,6524.000000,217.000000
50%,1.042478,1987.000000,3.773547e+06,45.000000,0.000000,15.727683,6771.000000,1.000000,6807.000000,82.000000,217.000000,29.000000,0.068106,103.000000,11339.500000,217.000000
75%,1.652872,2980.000000,3.773625e+06,68.000000,1.333422,23.405342,20055.000000,2.000000,21881.000000,125.000000,217.000000,44.000000,0.136133,103.000000,24191.750000,217.000000
max,28.474297,4806.000000,3.773695e+06,97.000000,3.141593,115.557260,105762.000000,2.000000,117025.000000,224.000000,1049.000000,59.000000,0.976192,103.000000,117025.000000,1049.000000


In [5]:
# Mostrar información general del DataFrame
df_events.info()

<class 'pandas.DataFrame'>
RangeIndex: 139030 entries, 0 to 139029
Columns: 111 entries, 50_50 to shot_follows_dribble
dtypes: float64(8), int64(8), object(59), str(36)
memory usage: 136.6+ MB


In [6]:
# Calcular el porcentaje de valores nulos por columna
porcentaje_nulos = df_events.isnull().sum() / len(df_events) * 100
print("Porcentaje de valores nulos por columna:")
print(porcentaje_nulos)

Porcentaje de valores nulos por columna:
50_50                             99.989930
bad_behaviour_card                99.982018
ball_receipt_outcome              96.716536
ball_recovery_offensive           99.996404
ball_recovery_recovery_failure    99.869812
                                    ...    
shot_saved_to_post                99.997842
block_save_block                  99.998561
dribble_no_touch                  99.998561
shot_redirect                     99.999281
shot_follows_dribble              99.999281
Length: 111, dtype: float64


Vemos que en todas las columnas el % de nulos enorme, esto se debe a que cada fila va asociada a un evento
osea una fila que sea 'Pass' tendra NaN en el resto de metricas.

In [7]:
df_events.columns

Index(['50_50', 'bad_behaviour_card', 'ball_receipt_outcome',
       'ball_recovery_offensive', 'ball_recovery_recovery_failure',
       'carry_end_location', 'clearance_aerial_won', 'clearance_body_part',
       'clearance_head', 'clearance_left_foot',
       ...
       'foul_won_penalty', 'goalkeeper_punched_out',
       'goalkeeper_shot_saved_off_target', 'goalkeeper_shot_saved_to_post',
       'shot_saved_off_target', 'shot_saved_to_post', 'block_save_block',
       'dribble_no_touch', 'shot_redirect', 'shot_follows_dribble'],
      dtype='str', length=111)

estos serian todos los eventos posibles en un partido

In [8]:
# Contar la cantidad de eventos por tipo
print(df_events['type'].value_counts())

type
Pass                 40337
Ball Receipt*        39197
Carry                32900
Pressure             10832
Ball Recovery         2822
Duel                  1763
Block                 1276
Dribble               1206
Clearance             1037
Goal Keeper            994
Foul Committed         953
Foul Won               909
Shot                   839
Interception           791
Dribbled Past          785
Miscontrol             708
Dispossessed           672
Substitution           300
Half Start             140
Half End               140
Injury Stoppage        111
Tactical Shift          79
Starting XI             70
Referee Ball-Drop       46
Bad Behaviour           25
Shield                  23
Error                   19
50/50                   14
Offside                 12
Player Off               8
Player On                8
Own Goal For             7
Own Goal Against         7
Name: count, dtype: int64


In [9]:
# Ver que otros eventos estan asociados a cada tipo de evento
MCO_atributos = ['Pass','Carry','Dribble','Pressure','Ball Receipt*','Miscontrol','Dispossessed','Duel',
                 'Shot','Ball Recovery',]
for atributo in MCO_atributos:
    print(f"Porcentaje de valores nulos para {atributo}:")
    porcentaje_nulos = df_events[df_events['type'] == atributo].isnull().mean()
    print(porcentaje_nulos[porcentaje_nulos < 0.8].sort_values(ascending=False))


Porcentaje de valores nulos para Pass:
pass_body_part        0.036964
pass_recipient_id     0.027940
pass_recipient        0.027940
related_events        0.004165
index                 0.000000
minute                0.000000
match_id              0.000000
pass_angle            0.000000
location              0.000000
id                    0.000000
duration              0.000000
pass_length           0.000000
pass_height           0.000000
pass_end_location     0.000000
period                0.000000
player                0.000000
player_id             0.000000
position              0.000000
play_pattern          0.000000
possession            0.000000
possession_team       0.000000
possession_team_id    0.000000
second                0.000000
team                  0.000000
team_id               0.000000
timestamp             0.000000
type                  0.000000
dtype: float64
Porcentaje de valores nulos para Carry:
under_pressure        0.72769
carry_end_location    0.00000
id       

Con esta celda vemos que cuando el tipo de evento es pase siempre tendremos informacion excepto a veces de quien la recibe, puede
que eso nos indique que nadie recibio el pase.

In [10]:
# Mira los eventos del final de un partido concreto
match_id_ejemplo = df_events['match_id'].unique()[0]
df_partido = df_events[df_events['match_id'] == match_id_ejemplo]
df_partido[df_partido['type'] == 'Half End'][['minute', 'period', 'timestamp']]

,minute,period,timestamp
3882,47,1,00:47:04.883
3883,47,1,00:47:04.883
3884,94,2,00:49:57.208
3885,94,2,00:49:57.208


minutos posibles dentro de un partido especifico

In [11]:
# Mira los eventos del final de un partido concreto relacionados a los cambios de jugadores

match_id_ejemplo1 = df_events['match_id'].unique()[1]
df_partido1 = df_events[df_events['match_id'] == match_id_ejemplo1]

eventos_tiempo=['Half End', 'Starting XI', 'Substitution']
df_eventos_tiempo=df_partido1.loc[df_partido1['type'].isin(eventos_tiempo), ['player', 'minute', 'period', 'timestamp','type', 'substitution_replacement']]
df_eventos_tiempo

,player,minute,period,timestamp,type,substitution_replacement
3891,NaN,0,1,00:00:00.000,Starting XI,NaN
3892,NaN,0,1,00:00:00.000,Starting XI,NaN
7645,NaN,46,1,00:46:04.284,Half End,NaN
7646,NaN,46,1,00:46:04.284,Half End,NaN
7647,NaN,91,2,00:46:53.099,Half End,NaN
7648,NaN,91,2,00:46:53.099,Half End,NaN
7649,Carlos Neva Tey,45,2,00:00:00.000,Substitution,Jesús Vallejo Lázaro
7650,Ángel Montoro Sánchez,45,2,00:00:00.000,Substitution,Luis Milla Manzanares
7651,Roberto Soldado Rillo,45,2,00:00:00.000,Substitution,Luis Javier Suárez Charris
7652,Darwin Daniel Machís Marcano,63,2,00:18:56.939,Substitution,Robert Kenedy Nunes do Nascimento


In [12]:
# Ver los eventos de tipo 'Starting XI' y 'Substitution' para el partido
df_partido1[df_partido1['type'] == 'Starting XI'].dropna(axis=1, how='all').columns.tolist()

['duration',
 'id',
 'index',
 'match_id',
 'minute',
 'period',
 'play_pattern',
 'possession',
 'possession_team',
 'possession_team_id',
 'second',
 'tactics',
 'team',
 'team_id',
 'timestamp',
 'type']

In [13]:
print(df_partido1[df_partido1['type'] == 'Substitution'].dropna(axis=1, how='all').columns.tolist())
df_partido1[df_partido1['type'] == 'Substitution']['substitution_replacement'].values

['duration', 'id', 'index', 'match_id', 'minute', 'period', 'play_pattern', 'player', 'player_id', 'position', 'possession', 'possession_team', 'possession_team_id', 'second', 'substitution_outcome', 'substitution_outcome_id', 'substitution_replacement', 'substitution_replacement_id', 'team', 'team_id', 'timestamp', 'type']


<ArrowStringArray>
[                            'Jesús Vallejo Lázaro',
                            'Luis Milla Manzanares',
                       'Luis Javier Suárez Charris',
                'Robert Kenedy Nunes do Nascimento',
                   'Martin Braithwaite Christensen',
                                   'Miralem Pjanić',
                             'Alberto Soro Álvarez',
 'Francisco António Machado Mota de Castro Trincão',
                                'Ricard Puig Martí',
                       'Héctor Junior Firpo Adames']
Length: 10, dtype: str

vemos los jugadores que entran al partido

In [14]:
#Sacar el 11 inicial de un partido concreto

df_partido1[df_partido1['type'] == 'Starting XI']['tactics'].values


array([{'formation': 4231.0, 'lineup': array([{'jersey_number': 1, 'player': {'id': 24140, 'name': 'Rui Tiago Dantas da Silva'}, 'position': {'id': 1, 'name': 'Goalkeeper'}},
              {'jersey_number': 2, 'player': {'id': 4546, 'name': 'Dimitri Foulquier'}, 'position': {'id': 2, 'name': 'Right Back'}},
              {'jersey_number': 22, 'player': {'id': 23807, 'name': 'Domingos Sousa Coutinho Meneses Duarte'}, 'position': {'id': 3, 'name': 'Right Center Back'}},
              {'jersey_number': 6, 'player': {'id': 24134, 'name': 'Germán Sánchez Barahona'}, 'position': {'id': 5, 'name': 'Left Center Back'}},
              {'jersey_number': 15, 'player': {'id': 25702, 'name': 'Carlos Neva Tey'}, 'position': {'id': 6, 'name': 'Left Back'}},
              {'jersey_number': 21, 'player': {'id': 16249, 'name': 'Yangel Clemente Herrera Ravelo'}, 'position': {'id': 13, 'name': 'Right Center Midfield'}},
              {'jersey_number': 8, 'player': {'id': 24052, 'name': 'Yan Brice Eteki'},

vemos que viene en un json que tendra que ser extraido

In [15]:
#crear df de los cambios para luego actualizar los minutos de juego de cada jugador
df_cambios = df_partido1[df_partido1['type'] == 'Substitution']
salen=df_cambios[['player', 'minute']].rename(columns={'player':'nombre','minute':'min_final'})
entran=df_cambios[['substitution_replacement', 'minute']].rename(columns={'substitution_replacement':'nombre','minute':'min_entrada'})

In [16]:
#Extraer los jugadores de la alineación inicial de ambos equipos

XIinicialA = df_partido1[df_partido1['type'] == 'Starting XI']['tactics'].values[0]
XIinicialB = df_partido1[df_partido1['type'] == 'Starting XI']['tactics'].values[1]

jugadoresXI_A = pd.DataFrame([XIinicialA.get('player',{}).get('name', '') for XIinicialA in XIinicialA.get('lineup', [])])


jugadoresXI_B = pd.DataFrame([XIinicialB.get('player',{}).get('name', '') for XIinicialB in XIinicialB.get('lineup', [])])


jugadoresXIunico=pd.concat([jugadoresXI_A, jugadoresXI_B], ignore_index=True).drop_duplicates()
jugadoresXIunico=jugadoresXIunico.rename(columns={0: 'nombre'})

# Crear un DataFrame con los jugadores únicos y sus minutos jugados
jugadores_completo = pd.concat([jugadoresXIunico, salen['nombre'], entran['nombre']], ignore_index=True).drop_duplicates().reset_index(drop=True)


minuto_final=df_partido1[df_partido1['type'] == 'Half End']['minute'].max()

jugadores_completo['minutes_played'] = minuto_final

jugadores_completo['match_id'] = match_id_ejemplo1

# Actualizar los minutos para los jugadores sustituidos
for _, row in salen.iterrows():
    jugadores_completo.loc[jugadores_completo['nombre']==row['nombre'], 'minutes_played'] = row['min_final']

for _, row in entran.iterrows():
    min_reales = minuto_final - row['min_entrada']
    jugadores_completo.loc[jugadores_completo['nombre']==row['nombre'], 'minutes_played'] = min_reales

jugadores_completo

,nombre,minutes_played,match_id
0,Rui Tiago Dantas da Silva,91,3773565
1,Dimitri Foulquier,91,3773565
2,Domingos Sousa Coutinho Meneses Duarte,91,3773565
3,Germán Sánchez Barahona,91,3773565
4,Carlos Neva Tey,45,3773565
5,Yangel Clemente Herrera Ravelo,66,3773565
6,Yan Brice Eteki,91,3773565
7,José Antonio Rodríguez Díaz,91,3773565
8,Ángel Montoro Sánchez,45,3773565
9,Darwin Daniel Machís Marcano,63,3773565


estos serian los jugadore que participaron en este partido

In [17]:
def minutos_totales_jugador(match_id, df_match):
    

    XIinicialA = df_match[df_match['type'] == 'Starting XI']['tactics'].values[0]
    XIinicialB = df_match[df_match['type'] == 'Starting XI']['tactics'].values[1]

    # Extraer los jugadores de la alineación inicial de ambos equipos
    datosXI_A = [
    {
        'nombre': jugador.get('player', {}).get('name', ''),
        'posicion': jugador.get('position', {}).get('name', '')
    } 
    for jugador in XIinicialA.get('lineup', [])
]
    datosXI_B = [
    {
        'nombre': jugador.get('player', {}).get('name', ''),
        'posicion': jugador.get('position', {}).get('name', '')
    } 
    for jugador in XIinicialB.get('lineup', [])
]

    #extraer los cambios de jugadores
    df_substituciones = df_match[df_match['type'] == 'Substitution']
    salen = df_substituciones[['player', 'minute',]].rename(columns={'player':'nombre', 'minute':'min_final'})
    entran = df_substituciones[['substitution_replacement', 'minute', 'position']].rename(columns={'substitution_replacement':'nombre', 'minute':'min_entrada', 'position':'posicion'})

    # Crear un DataFrame con los jugadores únicos y sus minutos jugados
    jugadores_completo = pd.concat([pd.DataFrame(datosXI_A), pd.DataFrame(datosXI_B), salen['nombre'], entran[['nombre','posicion']]], ignore_index=True).drop_duplicates().reset_index(drop=True)

    minuto_final = df_match[df_match['type'] == 'Half End']['minute'].max()
    jugadores_completo['minutes_played'] = minuto_final
    jugadores_completo['match_id'] = match_id

# Actualizar los minutos para los jugadores sustituidos
    jugadores_completo = jugadores_completo.set_index('nombre')
    salen = salen.set_index('nombre')
    salen = salen.rename(columns={'min_final': 'minutes_played'})
    entran = entran.set_index('nombre')
    min_reales = minuto_final - entran['min_entrada']
    entran['minutes_played'] = min_reales
    jugadores_completo.update(salen)
    jugadores_completo.update(entran)
    jugadores_completo = jugadores_completo.reset_index()

    jugadores_completo['posicion']= jugadores_completo['posicion'].fillna('Unknown')

    return jugadores_completo

In [18]:
#se hace a traves de una lsita para que no tengas que hacer concat con cada aprtido,
# se va metiendo cada partido en una lista y luego se concatena todos los partidos en un solo dataframe
todos_minutos = []

mins_prueba_agrupados=df_events.groupby('match_id')

for match_id, df_match in mins_prueba_agrupados:
    resultado = minutos_totales_jugador(match_id, df_match)
    todos_minutos.append(resultado)

df_minutos = pd.concat(todos_minutos, ignore_index=True)
print(f"Shape: {df_minutos.shape}")
print(df_minutos.values[11:25])



Shape: (1370, 4)
[['Edgar Badía Guardiola' 'Goalkeeper' 92 3764440]
 ['Antonio Barragán Fernández' 'Right Center Back' 92 3764440]
 ['Gonzalo Cacicedo Verdú' 'Center Back' 92 3764440]
 ['José Manuel Sánchez Guillén' 'Left Center Back' 92 3764440]
 ['Miguel Ángel Garrido Cifuentes' 'Right Wing Back' 59 3764440]
 ['Johan Andrés Mojica Palacio' 'Left Wing Back' 92 3764440]
 ['José Raúl Gutiérrez Parejo' 'Right Center Midfield' 92 3764440]
 ['Omenuke Mfulu' 'Left Center Midfield' 80 3764440]
 ['Pere Milla Peña' 'Right Wing' 71 3764440]
 ['Emiliano Ariel Rigoni' 'Left Wing' 59 3764440]
 ['Lucas Ariel Boyé' 'Center Forward' 71 3764440]
 ['Miralem Pjanić' 'Unknown' 45 3764440]
 ['Miguel Ángel Garrido Cifuentes' 'Unknown' 59 3764440]
 ['Emiliano Ariel Rigoni' 'Unknown' 59 3764440]]


In [19]:
# Agrupar por jugador y sumar los minutos jugados
df_total_minutos = df_minutos.groupby('nombre')['minutes_played'].sum().reset_index()
df_total_minutos = df_total_minutos.sort_values(by='minutes_played', ascending=False)


In [20]:
#hacer unstack para ver el numero de eventos por tipo de evento y por jugador
df_eventos_agrupados = df_events.groupby('player',)['type'].value_counts().unstack(fill_value=0)
df_eventos_agrupados.shape

(383, 27)

In [21]:
# Unir los DataFrames de eventos agrupados y minutos totales por jugador
df_completo=df_eventos_agrupados.join(df_total_minutos.set_index('nombre'), on='player', how='left').fillna(0).sort_values(by='minutes_played', ascending=False)
df_completo.head()  

,Pass,Goal Keeper,Carry,Ball Receipt*,Ball Recovery,Pressure,Duel,Dribbled Past,Clearance,Foul Committed,...,Injury Stoppage,Shield,Error,Offside,50/50,Own Goal Against,Bad Behaviour,Player Off,Player On,minutes_played
player,,,,,,,,,,,,,,,,,,,,,
Antoine Griezmann,1214,0,1016,1510,80,377,54,30,14,15,...,0,1,0,2,1,0,0,0,0,3785
Pedro González López,1644,0,1460,1716,121,638,55,46,4,23,...,4,0,0,0,0,0,0,0,0,3709
Sergio Busquets i Burgos,2534,0,1911,2282,103,496,102,35,22,46,...,8,0,2,0,0,0,0,0,0,3621
Jordi Alba Ramos,2815,0,2092,2510,140,335,60,38,43,22,...,3,0,0,0,1,1,6,0,0,3448
Frenkie de Jong,2583,0,2180,2524,123,337,60,24,29,27,...,4,0,1,1,2,0,0,0,0,3355


In [22]:
# Calcular las estadísticas por 90 minutos

minutos = df_completo['minutes_played']
df_completo_p90 = df_completo.div(minutos, axis=0) * 90
df_completo_p90 = df_completo_p90.round(2)
df_completo_p90['minutes_played'] = minutos
df_completo_p90.head()

,Pass,Goal Keeper,Carry,Ball Receipt*,Ball Recovery,Pressure,Duel,Dribbled Past,Clearance,Foul Committed,...,Injury Stoppage,Shield,Error,Offside,50/50,Own Goal Against,Bad Behaviour,Player Off,Player On,minutes_played
player,,,,,,,,,,,,,,,,,,,,,
Antoine Griezmann,28.87,0.0,24.16,35.90,1.90,8.96,1.28,0.71,0.33,0.36,...,0.00,0.02,0.00,0.05,0.02,0.00,0.00,0.0,0.0,3785
Pedro González López,39.89,0.0,35.43,41.64,2.94,15.48,1.33,1.12,0.10,0.56,...,0.10,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.0,3709
Sergio Busquets i Burgos,62.98,0.0,47.50,56.72,2.56,12.33,2.54,0.87,0.55,1.14,...,0.20,0.00,0.05,0.00,0.00,0.00,0.00,0.0,0.0,3621
Jordi Alba Ramos,73.48,0.0,54.61,65.52,3.65,8.74,1.57,0.99,1.12,0.57,...,0.08,0.00,0.00,0.00,0.03,0.03,0.16,0.0,0.0,3448
Frenkie de Jong,69.29,0.0,58.48,67.71,3.30,9.04,1.61,0.64,0.78,0.72,...,0.11,0.00,0.03,0.03,0.05,0.00,0.00,0.0,0.0,3355


In [23]:
# Guardar el DataFrame con las estadísticas por 90 minutos en un archivo Parquet para las 5 grandes ligas y luego cargarlo para no tener que volver a calcularlo cada vez
#se utilizan las estructuras de las celdas anteriores que eran de prueba para luego hacer un bucle con las 5 ligas y guardar el resultado global en un solo archivo parquet

import gc

archivos_ligas = [
    'data/raw/eventos_La_Liga_15_16.parquet',
    'data/raw/eventos_Bundesliga_15_16.parquet',
    'data/raw/eventos_Serie_A_15_16.parquet',
    'data/raw/eventos_Ligue_1_15_16.parquet',
    'data/raw/eventos_Premier_League_15_16.parquet',
]

mapeo_lineas_tacticas = {
    'Goalkeeper': 'POR',
    
    'Left Back': 'DEF', 'Right Back': 'DEF', 'Center Back': 'DEF', 
    'Left Center Back': 'DEF', 'Right Center Back': 'DEF',
    'Left Wing Back': 'DEF', 'Right Wing Back': 'DEF',
    
    'Center Defensive Midfield': 'MCD', 'Left Defensive Midfield': 'MCD', 'Right Defensive Midfield': 'MCD',
    
    'Center Midfield': 'MC', 'Left Center Midfield': 'MC', 'Right Center Midfield': 'MC', 'Right Midfield': 'MC', 'Left Midfield': 'MC',
    'Center Attacking Midfield': 'MCO', 'Left Attacking Midfield': 'MCO', 'Right Attacking Midfield': 'MCO',
    
    'Left Wing': 'DEL', 'Right Wing': 'DEL', 'Center Forward': 'DEL',
    'Secondary Striker': 'DEL', 'Left Center Forward': 'DEL', 'Right Center Forward': 'DEL'
}


if os.path.exists('data/processed/resumen_global_p90.parquet'):
    df_resumen_global_p90 = pd.read_parquet('data/processed/resumen_global_p90.parquet')
    print('Cargado resumen global desde el archivo local')
else:

    resumenes_p90_globales = []
    for archivo in archivos_ligas:
        print(f"Procesando archivo: {archivo}")
        df_events_liga = pd.read_parquet(archivo)

        partidos_agrupados = df_events_liga.groupby('match_id')

        minutos_partidos = []
        for match_id, df_match in partidos_agrupados:
            resumen_minutos = minutos_totales_jugador(match_id, df_match)
            minutos_partidos.append(resumen_minutos)
        df_minutos_liga = pd.concat(minutos_partidos, ignore_index=True)


        df_minutos_liga['posicion'] = df_minutos_liga['posicion'].map(mapeo_lineas_tacticas).fillna('Unknown')


        df_minutos_unicos = df_minutos_liga.groupby('nombre')['minutes_played'].sum().reset_index()
        df_posicion_principal = df_minutos_liga.groupby(['nombre','posicion'])['minutes_played'].sum().reset_index()
        df_posicion_principal = df_minutos_liga.sort_values(by='minutes_played', ascending=False).drop_duplicates(subset='nombre')[['nombre', 'posicion']]
        df_total_minutos_liga = pd.merge(df_minutos_unicos, df_posicion_principal, on='nombre')

        df_eventos_agrupados_liga = df_events_liga.groupby(['player', 'type']).size().unstack(fill_value=0)
        df_total_minutos_liga = df_total_minutos_liga.rename(columns={'nombre': 'player'}).set_index('player')

        df_completo_liga = df_eventos_agrupados_liga.join(df_total_minutos_liga, how='inner')

        minutos_liga = df_completo_liga['minutes_played']

        columnas_eventos = df_eventos_agrupados_liga.columns

        df_completo_p90_liga = df_completo_liga[columnas_eventos].div(minutos_liga, axis=0) * 90
        df_completo_p90_liga = df_completo_p90_liga.round(2)
        df_completo_p90_liga['minutes_played'] = minutos_liga
        df_completo_p90_liga['posicion'] = df_completo_liga['posicion']

        df_completo_p90_liga[columnas_eventos] = df_completo_p90_liga[columnas_eventos].fillna(0)
        df_completo_p90_liga = df_completo_p90_liga.reset_index().rename(columns={'index': 'player'})

        resumenes_p90_globales.append(df_completo_p90_liga)


        del df_events_liga, df_minutos_liga, df_total_minutos_liga, df_eventos_agrupados_liga, df_completo_liga, df_completo_p90_liga
        gc.collect()

    df_resumen_global_p90 = pd.concat(resumenes_p90_globales, ignore_index=True)
    df_resumen_global_p90.to_parquet('data/processed/resumen_global_p90.parquet', index=False)
    print(df_resumen_global_p90.head())


Cargado resumen global desde el archivo local


In [24]:
# Eliminar la columna 'Own Goal For' si existe, ya que no es relevante para el análisis de rendimiento de los jugadores
# Tiene muchos valores nulos y no aporta información sobre el rendimiento de los jugadores
df_resumen_global_p90.drop(columns=['Own Goal For'], inplace=True)

In [25]:
# Filtrar jugadores con al menos 500 minutos jugados para tener una muestra más representativa
df500mins=df_resumen_global_p90[df_resumen_global_p90['minutes_played'] >= 500]
df500mins.shape[0]

2070

In [26]:
# Filtrar jugadores con posición conocida para tener una muestra más representativa y evitar ruido en el análisis
df_final_scouting = df500mins[df500mins['posicion'] != 'Unknown'].copy()

In [27]:
import numpy as np
from sklearn.preprocessing import StandardScaler

# 1. Columnas que NO queremos estandarizar
columnas_omitir = ['player', 'posicion', 'minutes_played']

# 2. Nos quedamos solo con las columnas de eventos (las numéricas)
columnas_features = [col for col in df_final_scouting.columns if col not in columnas_omitir]

# === SOLUCIÓN: Limpiar los infinitos ocultos ===
# Creamos una copia de trabajo de las features numéricas
df_features_limpias = df_final_scouting[columnas_features].copy()

# Reemplazamos tanto [inf] como [-inf] por nulos, y luego los convertimos en 0
df_features_limpias = df_features_limpias.replace([np.inf, -np.inf], np.nan).fillna(0)
# ===============================================

# 3. Inicializamos el escalador
scaler = StandardScaler()

# 4. Ajustamos y transformamos la matriz ya limpia de infinitos
X_scaled = scaler.fit_transform(df_features_limpias)

# 5. Volvemos a empaquetar en un DataFrame limpio usando el mismo índice original
df_scaled = pd.DataFrame(X_scaled, columns=columnas_features, index=df_final_scouting['player'])

print("¡Estandarización completada con éxito! Filas procesadas:", df_scaled.shape[0])

¡Estandarización completada con éxito! Filas procesadas: 2030


In [28]:

df_scaled.head()

,Bad Behaviour,Ball Receipt*,Ball Recovery,Block,Carry,Clearance,Dispossessed,Dribble,Dribbled Past,Duel,...,Shield,Shot,50/50,Substitution,Offside,Player Off,Player On,Error,Goal Keeper,Own Goal Against
player,,,,,,,,,,,,,,,,,,,,,
Abdoul Karim Yoda,1.462352,0.456597,1.615977,0.399811,1.189884,-0.043781,0.863860,4.515594,3.005630,-0.218133,...,0.408911,-0.252781,-0.988692,-1.314511,-0.405754,-0.684928,-0.683926,-0.453776,-0.271949,-0.223387
Abdoulaye Doucouré,2.637374,0.121503,2.750753,0.674294,0.711196,-0.402581,0.193437,0.089944,0.832426,0.503196,...,-0.694604,-0.225390,-0.453419,-0.889887,-0.405754,-0.684928,-0.683926,-0.453776,-0.271949,-0.223387
Abraham González Casanova,-0.652687,-0.391191,-0.556308,-0.163601,-0.483609,-0.766737,-0.736947,-0.832066,0.304949,-0.353828,...,-0.694604,0.021130,-0.988692,1.111911,-0.405754,-0.684928,-0.683926,-0.453776,-0.271949,-0.223387
Adalberto Peñaranda Maestre,-0.652687,-0.412134,-1.002112,0.009757,-0.742101,-0.884552,1.903701,1.206062,-0.496816,0.931708,...,-0.204153,-0.102130,-0.988692,1.172572,0.874246,0.031775,0.039183,-0.453776,-0.271949,-0.223387
Aderllan Leandro de Jesus Santos,-0.652687,-0.464912,-0.677891,0.674294,-0.038429,1.760931,-0.969543,-0.667075,-0.159230,0.596040,...,2.125490,-0.581476,-0.988692,-1.314511,-0.405754,-0.684928,-0.683926,-0.453776,-0.271949,-0.223387


In [29]:
df_messi_scaled = df_scaled.loc['Lionel Andrés Messi Cuccittini']
df_messi_scaled.head()

Bad Behaviour    0.052326
Ball Receipt*    3.602293
Ball Recovery   -0.775157
Block           -1.622693
Carry            3.740333
Name: Lionel Andrés Messi Cuccittini, dtype: float64

In [30]:
# Ahora sí, podemos usar el modelo KNN para encontrar los jugadores más similares a Messi en base a sus estadísticas estandarizadas por 90 minutos
from sklearn.neighbors import NearestNeighbors
    
modelo_knn = NearestNeighbors(n_neighbors=10)
modelo_knn.fit(df_scaled[columnas_features])

distancias, indices = modelo_knn.kneighbors([df_messi_scaled[columnas_features]])


In [31]:
# Mostrar los jugadores similares a Messi y sus distancias
jugadores_similares = df_scaled.index[indices[0]]
print(pd.DataFrame({'Jugador Similar': jugadores_similares, 'Distancia': distancias[0]}))

                    Jugador Similar  Distancia
0    Lionel Andrés Messi Cuccittini   0.000000
1     Neymar da Silva Santos Junior   5.586325
2  Alexis Alejandro Sánchez Sánchez   7.006824
3                       Eden Hazard   7.319234
4              Jonathan Viera Ramos   7.552046
5                  Keita Baldé Diao   7.713543
6                     Dries Mertens   7.807204
7                      Ross Barkley   7.878903
8                    Hatem Ben Arfa   7.940723
9                     Memphis Depay   7.979146


In [32]:
#funcion que productiviza el modelo KNN para encontrar jugadores similares a cualquier jugador del dataset, 
# no solo Messi, y que además filtre por posición para comparar solo con jugadores de la misma posición y evitar ruido en el análisis

def find_similar_players(player_name, n_neighbors=10):
    # Verificar si el jugador existe en el DataFrame
    if player_name not in df_final_scouting['player'].values:
        print(f"Jugador '{player_name}' no encontrado en el dataset.")
        return None
    #sacar la posición del jugador para luego comparar solo con jugadores de la misma posición
    info_jugador = df_final_scouting[df_final_scouting['player'] == player_name].iloc[0]
    posicion_jugador = info_jugador['posicion']
    # Filtrar el DataFrame para quedarnos solo con jugadores de la misma posición
    df_posicion = df_final_scouting[df_final_scouting['posicion'] == posicion_jugador].copy()

    # 1. Columnas que NO queremos estandarizar
    columnas_omitir = ['player', 'posicion', 'minutes_played']
    columnas_features = [col for col in df_posicion.columns if col not in columnas_omitir]

    df_features_limpias = df_posicion[columnas_features].replace([np.inf, -np.inf], np.nan).fillna(0)

    # 2. Estandarizamos solo las columnas numéricas
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df_features_limpias)

    df_scaled = pd.DataFrame(X_scaled, columns=columnas_features, index=df_posicion['player'])

    # 3. Creamos el modelo de vecinos más cercanos y lo ajustamos con los datos estandarizados
    nn = NearestNeighbors(n_neighbors=n_neighbors + 1, metric='cosine')
    nn.fit(df_scaled)

    # 4. Obtenemos el vector del jugador objetivo y calculamos sus vecinos más cercanos
    vector_jugador = df_scaled.loc[[player_name]]

    distancias, indices = nn.kneighbors(vector_jugador)

    # 5. Preparamos un DataFrame con los resultados, incluyendo la posición y minutos jugados de cada vecino
    resultados = []
    for i in range(len(indices[0])):
        nombre_vecino = df_scaled.index[indices[0][i]]
       # Excluir al jugador objetivo de los resultados 
        if nombre_vecino==player_name:
            continue
            # Obtener la posición y minutos jugados del vecino desde el DataFrame original
        datos_originales = df_posicion[df_posicion['player'] == nombre_vecino].iloc[0]
        resultados.append({
            'Jugador Similar': nombre_vecino,
            'Posición': datos_originales['posicion'],
            'Minutos Jugados': datos_originales['minutes_played'],
            'Distancia Coseno': distancias[0][i]
        })

    df_reporte = pd.DataFrame(resultados).sort_values(by='Distancia Coseno')
    print(f"Jugadores similares a '{player_name}' (Posición: {posicion_jugador}):")

    return df_reporte.head(n_neighbors)



In [33]:
find_similar_players('Lionel Andrés Messi Cuccittini', n_neighbors=10)

Jugadores similares a 'Lionel Andrés Messi Cuccittini' (Posición: DEL):


,Jugador Similar,Posición,Minutos Jugados,Distancia Coseno
0,Neymar da Silva Santos Junior,DEL,3215,0.134113
1,Alexis Alejandro Sánchez Sánchez,DEL,3449,0.194140
2,Keita Baldé Diao,DEL,2173,0.209009
3,Eden Hazard,DEL,2648,0.212389
4,Ángel Fabián Di María Hernández,DEL,3038,0.272193
5,Kingsley Coman,DEL,2164,0.273854
6,Nabil Fekir,DEL,636,0.279371
7,Franck Bilal Ribéry,DEL,763,0.325633
8,Dries Mertens,DEL,1521,0.333111
9,Cristiano Ronaldo dos Santos Aveiro,DEL,3397,0.338915


como es normal vemos que los jugadores "mas" parecidos a messi son neymar y alexis sanchez, jugadores que en esta temporada, como messi, eran el centro de creatividad y peligro en sus equipos recibiendo la pelote en el ultimo tercio del campo, regateando, creando oportunidades para sus compañeros y para ellos mismos.


In [34]:
# Guardar el DataFrame final de scouting con las estadísticas por 90 minutos para los jugadores con al menos 500 minutos jugados 
# y posición conocida en un archivo Parquet para futuras consultas sin tener que recalcular todo el proceso
df_final_scouting.to_parquet('data/processed/df_final_scouting.parquet', index=False)